# 🎨 Notebook 1: Blackjack — Class Design


## 🛠️ Setup

```bash
cd 07-object-oriented-design/blackjack
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 What you'll learn

We'll design a Blackjack game step by step, moving from a **bad** one-function script to a **clean** object-oriented design.

Blackjack is a perfect small OOD problem: it has a clear domain (cards, deck, hand, player), one rule engine (deal / hit / stand / bust), and natural polymorphism (Player vs Dealer).

> 💡 If you've never played Blackjack: each card has a point value, you get dealt two cards, and you keep asking for more (*hit*) or stop (*stand*). Whoever gets closest to 21 without going over wins. The Ace counts as 11 or 1 — your choice.


## 🚫 Bad design: everything in one giant function

First, let's see what happens when we *don't* think in objects. This works, but every new feature means editing this one tangled function.


In [ ]:
import random

def play_blackjack_badly(seed=0):
    random.seed(seed)
    # Deck as a list of (rank, suit) tuples — no class at all
    ranks = ["A","2","3","4","5","6","7","8","9","10","J","Q","K"]
    suits = ["♥","♦","♣","♠"]
    deck = [(r, s) for s in suits for r in ranks]
    random.shuffle(deck)

    player = [deck.pop(), deck.pop()]
    dealer = [deck.pop(), deck.pop()]

    # Scoring logic is duplicated every time we need it 😬
    def score(hand):
        total, aces = 0, 0
        for r, _ in hand:
            if r == "A": total += 11; aces += 1
            elif r in ("J","Q","K"): total += 10
            else: total += int(r)
        while total > 21 and aces:
            total -= 10; aces -= 1
        return total

    # Player hits until 17 (duplicated rule — also used for dealer below)
    while score(player) < 17:
        player.append(deck.pop())
    while score(dealer) < 17:
        dealer.append(deck.pop())

    ps, ds = score(player), score(dealer)
    print(f"Player {player} = {ps}")
    print(f"Dealer {dealer} = {ds}")
    if ps > 21:          print("Player busts — dealer wins")
    elif ds > 21 or ps>ds: print("Player wins")
    elif ps == ds:       print("Push")
    else:                print("Dealer wins")

play_blackjack_badly()


### 🔴 What's wrong with this code?

| Smell | Why it hurts |
|---|---|
| **One huge function** | Hard to read, hard to test, hard to change. |
| **Magic tuples** `(rank, suit)` | No single place owns *"what is a card?"* |
| **Duplicated scoring** | If we later add *splitting* or *soft-17*, we edit in many places. |
| **Player and Dealer share code by copy-paste** | No polymorphism — can't swap strategies. |
| **No seams for tests** | We can't test the Ace rule without running a whole game. |

These are the classic symptoms that tell you: *"this wants to be objects."*


## ✅ Better design: one class per concept

We identify the **nouns** of the domain and give each one a class with a single responsibility:

```
┌────────┐ 1    * ┌──────┐
│  Deck  │◆──────│ Card │    Deck owns Cards
└────────┘        └──────┘
   │ draws
   ▼
┌──────┐   *   1 ┌────────┐
│ Hand │◄───────│ Player │   Player has a Hand
└──────┘         └────────┘
                      △
                      │  inherits
                  ┌────────┐
                  │ Dealer │  Dealer is a special Player
                  └────────┘

┌──────┐ 1   1..* ┌────────┐
│ Game │◆────────│ Player │   Game orchestrates Players + Dealer
└──────┘          └────────┘
```

| Class | Responsibility (single!) |
|---|---|
| `Card` | Knows its rank, suit, and point value |
| `Deck` | Holds 52 cards, shuffles, deals |
| `Hand` | A list of cards + the Ace-aware `value()` rule |
| `Player` | Decides *hit* vs *stand*, owns one Hand |
| `Dealer` | A `Player` with the casino rule: hit until ≥ 17 |
| `Game` | Runs the round and settles wins/losses |


### 🧱 Skeletons — just the shape, no logic yet

Let's sketch the classes so the *structure* is obvious before we fill in the rules. The real logic comes in Notebook 2.


In [ ]:
from dataclasses import dataclass, field
from enum import Enum

class Suit(Enum):
    HEARTS = "♥"; DIAMONDS = "♦"; CLUBS = "♣"; SPADES = "♠"

@dataclass(frozen=True)
class Card:
    rank: str
    suit: Suit
    def value(self) -> int: ...   # filled in Notebook 2

@dataclass
class Hand:
    cards: list = field(default_factory=list)
    def add(self, c): ...
    def value(self) -> int: ...   # the Ace rule lives HERE — one place

class Player:
    def __init__(self, name): self.name, self.hand = name, Hand()
    def wants_hit(self) -> bool: ...   # strategy decision

class Dealer(Player):                  # polymorphism!
    def wants_hit(self) -> bool: ...   # casino rule: hit while < 17

print("Skeletons defined. Each class has one job.")


### 🧠 The tricky rule: the Ace

An Ace is worth **11 or 1**, whichever keeps you from busting. Where should that logic live?

- ❌ In `Player`? No — every player would duplicate it.
- ❌ In `Game`? No — that mixes rules with flow control.
- ✅ In `Hand.value()` — a Hand *is* "some cards plus the rule that totals them."

This is the **Single Responsibility Principle** (the `S` in SOLID) in one sentence:
*each class should have one reason to change.*


### 🔄 Why `Dealer` inherits from `Player`

Both need: a name, a hand, and a `wants_hit()` decision. The *only* thing that differs is the strategy. So `Dealer` overrides `wants_hit()` — this is **polymorphism**: `Game` can treat every seat at the table the same way.

> 🚧 Later, when we want 5 different playing styles, inheritance becomes clumsy. Notebook 3 swaps it for the **Strategy Pattern** — *composition over inheritance*.


## ➡️ Next

Open **`02_implementation.ipynb`** to fill in the bodies and watch a full round play out.
